In [2]:
import pandas as pd

df = pd.read_csv("../data/loan_approval_dataset.csv")

df.columns = df.columns.str.strip()


In [3]:
df = df.drop(columns=['loan_id'])

In [6]:
asset_columns = [
    'residential_assets_value',
    'commercial_assets_value',
    'luxury_assets_value',
    'bank_asset_value'
]

for col in asset_columns:
    print(col)
    print((df[col] < 0).sum())

residential_assets_value
28
commercial_assets_value
0
luxury_assets_value
0
bank_asset_value
0


In [7]:
for col in asset_columns:

    negative_count = (df[col] < 0).sum()

    if negative_count > 0:

        median_val = df.loc[
            df[col] >= 0,
            col
        ].median()

        df[col] = df[col].apply(
            lambda x: median_val if x < 0 else x
        )

        print(f"{col}: {negative_count} negatives replaced")

residential_assets_value: 28 negatives replaced


In [8]:
df['education'] = (
    df['education']
    .str.strip()
    .map({
        'Graduate': 1,
        'Not Graduate': 0
    })
)

df['self_employed'] = (
    df['self_employed']
    .str.strip()
    .map({
        'Yes': 1,
        'No': 0
    })
)

df['loan_status'] = (
    df['loan_status']
    .str.strip()
    .map({
        'Approved': 1,
        'Rejected': 0
    })
)

In [9]:
df['total_assets'] = (
    df['residential_assets_value']
    + df['commercial_assets_value']
    + df['luxury_assets_value']
    + df['bank_asset_value']
)

In [10]:
df['loan_to_income_ratio'] = (
    df['loan_amount']
    / (df['income_annum'] + 1)
)

In [11]:
df['annual_payment'] = (
    df['loan_amount']
    / df['loan_term']
)

In [12]:
df['debt_service_ratio'] = (
    df['annual_payment']
    / (df['income_annum'] + 1)
)

In [13]:
df['assets_to_loan_ratio'] = (
    df['total_assets']
    / (df['loan_amount'] + 1)
)

In [14]:
df['income_per_dependent'] = (
    df['income_annum']
    / (df['no_of_dependents'] + 1)
)

In [15]:
X = df.drop(columns=['loan_status'])
y = df['loan_status']

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

In [18]:
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [19]:
y_pred = pipeline.predict(X_test)

In [20]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9461358313817331
F1 Score: 0.9568480300187617
[[298  25]
 [ 21 510]]
              precision    recall  f1-score   support

           0       0.93      0.92      0.93       323
           1       0.95      0.96      0.96       531

    accuracy                           0.95       854
   macro avg       0.94      0.94      0.94       854
weighted avg       0.95      0.95      0.95       854



In [25]:
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [27]:
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [28]:
y_pred = pipeline.predict(X_test)

In [29]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9964871194379391
F1 Score: 0.9971830985915493
[[320   3]
 [  0 531]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       323
           1       0.99      1.00      1.00       531

    accuracy                           1.00       854
   macro avg       1.00      1.00      1.00       854
weighted avg       1.00      1.00      1.00       854



In [30]:
from xgboost import XGBClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [31]:
pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None


In [32]:
y_pred = pipeline.predict(X_test)

In [33]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.9964871194379391
F1 Score: 0.9971777986829727
[[321   2]
 [  1 530]]
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       323
           1       1.00      1.00      1.00       531

    accuracy                           1.00       854
   macro avg       1.00      1.00      1.00       854
weighted avg       1.00      1.00      1.00       854

